# Temporal Analysis of misinformation and informative messages

In [12]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import powerlaw, truncnorm, ks_2samp
import powerlaw



In [2]:
# Load the datasets
df = pd.read_json("../../Datasets/voluntariosdanavalencia_old.json") # Import data from Voluntarios de Valencia groupchat.
df_links = pd.read_csv("../../Datasets/voluntariosdanavalencia_extracted_links.csv") # Import data from Voluntarios de Valencia groupchat with messages with links and misinformation score.
df_users = pd.read_csv("../../Datasets/voluntariosdanavalencia_extracted_users.csv") # Import data from Voluntarios de Valencia groupchat with user information.

## Extract misinformation of messages

In [3]:
# Check possible values for fake_information column in df_links
print(df_links['fake_information'].unique())

<StringArray>
[             nan,        'unknown',           'high',          'Mixed',
            'low',           'True', 'mostly factual',      'very high',
          'mixed',       'very-low']
Length: 10, dtype: str


In [4]:
# Gather the messages with misinformation
misinformation_levels = ['Mixed', 'low', 'True', 'mixed', 'very-low']
# Gather messages with those values in the "fake information" column
misinformation_messages = df_links[df_links['fake_information'].isin(misinformation_levels)]
# Gather messages with "high" or "very-high" misinformation score
informative_messages = df_links[df_links['fake_information'].isin(['high', 'very high', np.nan])] # idk if I should add nan values, but I will do it just in case



Try to gather the misinformation & informative messages from the misinformation dataset into the df dataframe.

In [10]:
# Work on copies of the datasets to avoid modifying the original ones
mis = misinformation_messages.copy()
inf = informative_messages.copy()
full = df.copy()

# Normalize sender_id type
mis["sender_id"]= pd.to_numeric(mis["sender_id"], errors='coerce').astype('Int64')
inf["sender_id"]= pd.to_numeric(inf["sender_id"], errors='coerce').astype('Int64')
full["sender_id"]= pd.to_numeric(full["sender_id"], errors='coerce').astype('Int64')

# normalize date to UTC datatime
mis["date_dt"] = pd.to_datetime(mis["date"], errors='coerce', utc=True)
inf["date_dt"] = pd.to_datetime(inf["date"], errors='coerce', utc=True)
full["date_dt"] = pd.to_datetime(full["date"], errors='coerce', utc=True)

# Exact match of sender_id in df_users and mis/inf datasets and date
matches_mis = mis.merge(
    full,
    on=["sender_id", "date_dt"],
    how="left",
    suffixes=("_mis","_df")
)
matches_inf = inf.merge(
    full,
    on=["sender_id", "date_dt"],
    how="left",
    suffixes=("_inf","_df")
)   

# Found vs not found matches
found_mis = matches_mis[~matches_mis['id'].isna()]
not_found_mis = matches_mis[matches_mis['id'].isna()]
found_inf = matches_inf[~matches_inf['id'].isna()]
not_found_inf = matches_inf[matches_inf['id'].isna()]

print(f"Found matches in misinformation messages: {len(found_mis)}")
print(f"Found matches in informative messages: {len(found_inf)}")
print(f"Not found matches in misinformation messages: {len(not_found_mis)}")
print(f"Not found matches in informative messages: {len(not_found_inf)}")



Found matches in misinformation messages: 182
Found matches in informative messages: 1954
Not found matches in misinformation messages: 0
Not found matches in informative messages: 7


In [11]:

# Preview matches rows head
print("Misinformation found matches preview:")
found_mis[["date_dt", "sender_id", "links", "id"]].head()

Misinformation found matches preview:


,date_dt,sender_id,links,id
0,2024-11-02 07:49:30+00:00,702737014,https://x.com/okdiario/status/1852342806922043...,8361
1,2024-11-02 08:53:47+00:00,524281487,https://gaceta.es/espana/mazon-y-marlaska-pide...,8556
2,2024-11-02 10:15:06+00:00,1696509502,https://t.me/rubengisbertoficial/2008,8925
3,2024-11-02 12:27:53+00:00,7046448,https://www.larazon.es/sociedad/acertada-predi...,9660
4,2024-11-02 16:33:21+00:00,7094531229,https://cnnespanol.cnn.com/2024/11/01/inundaci...,11195


In [8]:
print("\nInformative found matches preview:")
found_inf[["date_dt", "sender_id", "links", "id"]].head()


Informative found matches preview:


,date_dt,sender_id,links,id
0,2024-10-30 15:00:42+00:00,1110241832,https://www.instagram.com/victimas_dana_vlc/pr...,47.0
1,2024-10-30 19:15:18+00:00,1110241832,https://www.google.com/maps/d/viewer?mid=1WbP4...,50.0
2,2024-10-30 19:58:22+00:00,1110241832,https://g.co/kgs/y7dLSiz,56.0
3,2024-10-30 19:59:30+00:00,1110241832,https://g.co/kgs/q6e1mmU,57.0
4,2024-10-30 20:00:34+00:00,1110241832,https://g.co/kgs/VDFK7XY,58.0


## Temporal analysis of messafes one hour after:

### Misinformation links

In [ ]:
def prepare_full_events(df):
    """Prepare the full events DataFrame by converting 'id' and 'sender_id' to numeric types, and 'date' to datetime format. Rows with invalid data will be dropped."""
    full_copy = df.copy() # Create a copy of the original DataFrame to avoid modifying it
    full_copy["id"]=pd.to_numeric(full_copy["id"], errors='coerce').astype('Int64') # Convert 'id' to numeric, coercing errors to NaN, and then to Int64
    full_copy["sender_id"]= pd.to_numeric(full_copy["sender_id"], errors='coerce').astype('Int64') # Convert 'sender_id' to numeric, coercing errors to NaN, and then to Int64
    full_copy["date_dt"] = pd.to_datetime(full_copy["date"], errors='coerce', utc=True) # Convert 'date' to datetime, coercing errors to NaT, and ensuring UTC timezone
    full_copy = full_copy.dropna(subset=[["id", "sender_id", "date_dt"]]).copy() # Drop rows with NaN values in 'id', 'sender_id', or 'date_dt' and create a copy of the resulting DataFrame
    return full_copy

def build_link_id_lookup(links_df, full_df):
    """Build a lookup DataFrame by merging the links DataFrame with the full events DataFrame on 'sender_id' and 'date_dt'. Only rows with valid matches will be retained."""
    links_copy = links_df.copy() # Create a copy of the links DataFrame to avoid modifying the original
    links_copy["sender_id"]= pd.to_numeric(links_copy["sender_id"], errors='coerce').astype('Int64') # Convert 'sender_id' to numeric, coercing errors to NaN, and then to Int64
    links_copy["date_dt"] = pd.to_datetime(links_copy["date"], errors='coerce', utc=True) # Convert 'date' to datetime, coercing errors to NaT, and ensuring UTC timezone
    full_lookup =full_df[["id", "sender_id", "date_dt"]].drop_duplicates().copy() # Create a copy of the relevant columns from the full DataFrame for lookup, dropping duplicates
    merged = links_copy.merge(
        full_lookup, on=["sender_id", "date_dt"], how="left", suffixes=("_links", "_full")
    ) # Merge the links DataFrame with the full lookup DataFrame on 'sender_id'
    merged = merged.dropna(subset=["id"]).copy() # Drop rows where 'id' is NaN and create a copy of the resulting DataFrame
    merged["id"] = merged["id"].astype('Int64') # Convert 'id' to Int64 type after dropping NaN values
    return merged

def collect_waiting_times(full_df, sources_df, window_minutes=60, mode="all"):
    
    if mode not in ["all", "first"]
        raise ValueError("Mode must be either 'all' or 'first'.") # Validate mode input
    full_local= full_df.copy() # Create a copy of the full DataFrame to avoid modifying the original
    sources_local = sources_df[["id", "date_dt", "topic_name"]].dropna().drop_duplicates(subset=["id"]).copy() # Create a copy of the relevant columns from the sources DataFrame, dropping rows with NaN values and duplicates based on 'id'

    window = pd.Timedelta(minutes=window_minutes) # Define the time window for waiting time calculation
    waiting_times = [] # Initialize a list to store waiting times

    for _, src in sources_local.iterrows(): #Iterate over each source event in the sources DataFrame
        same_topic = full_local[full_local["topic_name"] == src["topic_name"]].copy() # Filter the full DataFrame to get events with the same topic as the source event
        in_window = same_topic[
            (same_topic["date_dt"] >= src["date_dt"]) & (same_topic["date_dt"] <= src["date_dt"] + window)
        ].copy() # Filter the same topic events to get those that occur within the defined time window after the source event
        dt = full_local.loc[same_topic & in_window, "date_dt"].sort_values()   # Get the 'date_dt' values of the events that are in the same topic and within the time window, sorted by date
        if not in_window.empty: # Check if there are any events in the time window
            if mode == "first":
                first_event = in_window.sort_values("date_dt").iloc[0] # Get the first event in the time window based on date
                waiting_time = (first_event["date_dt"] - src["date_dt"]).total_seconds()  # Calculate waiting time in seconds
                waiting_times.append(waiting_time) # Append the waiting time to the list
            else:
                deltas = 